In [87]:
# check install for not common libraries 
# !pip install yfinance


In [88]:
# making sure API key is hidden 

In [89]:
import databento as db 
import pandas as pd 
import numpy as np 
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

from dotenv import load_dotenv



load_dotenv()
client = db.Historical()



In [90]:
def fetch_quantitative_data(ticker_symbol):
    print(f"📡 Fetching 15-minute data for {ticker_symbol}...")
    
    ticker_symbol = ticker_symbol.replace('.', '-')
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="60d", interval="15m")
    
    if df.empty:
        raise ValueError(f"No data found for {ticker_symbol}. Check ticker symbol.")
        
    # --- THE FIX FOR yfinance's NEW UPDATE ---
    # If yfinance returned multi-level columns, this flattens them back to normal
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    # -----------------------------------------
        
    df.index = df.index.tz_convert('America/New_York')
    
    df['Typical_Price'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['Price_Volume'] = df['Typical_Price'] * df['Volume']
    
    df['Date'] = df.index.date
    df['Cumulative_PV'] = df.groupby('Date')['Price_Volume'].cumsum()
    df['Cumulative_Volume'] = df.groupby('Date')['Volume'].cumsum()
    
    df['VWAP'] = df['Cumulative_PV'] / df['Cumulative_Volume']
    df['VWAP_Distance_Pct'] = round(((df['Close'] - df['VWAP']) / df['VWAP']) * 100, 3)
    
    df['Return'] = df['Close'].pct_change()
    rolling_mean = df['Return'].rolling(window=26).mean()
    rolling_std = df['Return'].rolling(window=26).std()
    df['Z_Score'] = round((df['Return'] - rolling_mean) / rolling_std, 3)
    
    rolling_vol_avg = df['Volume'].rolling(window=26).mean()
    df['Volume_Shock'] = round(df['Volume'] / (rolling_vol_avg + 1e-9), 2)
    
    df = df.dropna()
    cols_to_keep = ['Close', 'Volume', 'VWAP', 'VWAP_Distance_Pct', 'Z_Score', 'Volume_Shock']
    df_clean = df[cols_to_keep].rename(columns={'Close': 'Price'})
    
    print(f"✅ Successfully loaded {len(df_clean)} intervals.")
    return df_clean

# Test the newly fixed function!
quant_data = fetch_quantitative_data("BRK-B")
display(quant_data.tail(15))

📡 Fetching 15-minute data for BRK-B...
✅ Successfully loaded 1495 intervals.


,Price,Volume,VWAP,VWAP_Distance_Pct,Z_Score,Volume_Shock
Datetime,,,,,,
2026-04-20 12:15:00-04:00,472.750000,118563,474.161492,-0.298,0.766,0.71
2026-04-20 12:30:00-04:00,472.750092,124223,474.086602,-0.282,0.528,0.74
2026-04-20 12:45:00-04:00,472.954987,97001,474.039834,-0.229,0.967,0.58
2026-04-20 13:00:00-04:00,473.388611,113173,474.001797,-0.129,1.405,0.68
2026-04-20 13:15:00-04:00,473.440002,64601,473.989122,-0.116,0.487,0.39
2026-04-20 13:30:00-04:00,472.834991,130712,473.942082,-0.234,-1.022,0.79
2026-04-20 13:45:00-04:00,472.570007,82922,473.902642,-0.281,-0.210,0.50
2026-04-20 14:00:00-04:00,472.600006,125837,473.840697,-0.262,0.417,0.76
2026-04-20 14:15:00-04:00,472.934998,98257,473.808174,-0.184,1.147,0.59


In [91]:
# 1. Ask the user for the ticker and clean the input
# .strip() removes accidental spaces, .upper() ensures it matches API formatting
target_ticker = input("Enter a stock ticker (e.g., AAPL, TSLA, BRK-B): ").strip().upper()

# 2. Safety check to ensure the user actually typed something
if target_ticker:
    print(f"\n Initializing Quantitative Engine for {target_ticker}...\n")
    
    try:
        # 3. Feed the user's input directly into our data pipeline
        quant_data = fetch_quantitative_data(target_ticker)
        
        # 4. Display the resulting mathematical dataframe
        print(f"\n--- {target_ticker} Data Loaded ---")
        display(quant_data.head(25))
        
    except Exception as e:
        print(f" Error pulling data for {target_ticker}: {e}")
        print("Make sure you used a valid ticker symbol (use dashes for class shares, e.g., BRK-B).")
else:
    print(" No ticker entered. Please run the cell again.")


 Initializing Quantitative Engine for QQQ...

📡 Fetching 15-minute data for QQQ...
✅ Successfully loaded 1495 intervals.

--- QQQ Data Loaded ---


,Price,Volume,VWAP,VWAP_Distance_Pct,Z_Score,Volume_Shock
Datetime,,,,,,
2026-01-26 09:30:00-05:00,624.815002,2754300,624.002502,0.130,2.328,1.87
2026-01-26 09:45:00-05:00,625.294922,1327795,624.378454,0.147,0.414,0.90
2026-01-26 10:00:00-05:00,625.320007,1047939,624.529596,0.127,-0.150,0.72
2026-01-26 10:15:00-05:00,625.489990,4793386,625.085281,0.065,0.104,3.04
2026-01-26 10:30:00-05:00,625.330017,446322,625.107292,0.036,-0.304,0.29
2026-01-26 10:45:00-05:00,626.335022,834224,625.184062,0.184,1.344,0.54
2026-01-26 11:00:00-05:00,626.450012,799196,625.263468,0.190,0.076,0.52
2026-01-26 11:15:00-05:00,626.900024,795191,625.340711,0.249,0.494,0.52
2026-01-26 11:30:00-05:00,626.719971,793653,625.434484,0.206,-0.380,0.52


In [92]:
def generate_dynamic_signals(df, target_trades_per_hour=2, base_z_threshold=2.5):
    # Create a copy so we don't mess up our original data
    signals_df = df.copy()
    
    # Initialize our new tracking columns
    signals_df['Signal'] = 'HOLD'
    signals_df['Urgency'] = 1.0
    signals_df['Adjusted_Thresh'] = base_z_threshold
    
    trades_this_hour = 0
    current_hour = -1
    
    # We iterate row-by-row because the logic is path-dependent
    for i in range(len(signals_df)):
        row = signals_df.iloc[i]
        timestamp = signals_df.index[i]
        
        # 1. RESET THE CLOCK EVERY HOUR
        if timestamp.hour != current_hour:
            trades_this_hour = 0
            current_hour = timestamp.hour
            
        # 2. CALCULATE URGENCY
        intervals_passed = (timestamp.minute // 15) + 1 
        expected_trades_so_far = (intervals_passed / 4.0) * target_trades_per_hour
        
        urgency = 1.0
        if trades_this_hour < expected_trades_so_far:
            urgency = 1.0 + (expected_trades_so_far - trades_this_hour) * 0.5
        elif trades_this_hour >= target_trades_per_hour:
            urgency = 0.2 
            
        # 3. CALCULATE THE ADJUSTED THRESHOLD
        current_thresh = round(base_z_threshold / urgency, 3)
        
        signals_df.iloc[i, signals_df.columns.get_loc('Urgency')] = round(urgency, 2)
        signals_df.iloc[i, signals_df.columns.get_loc('Adjusted_Thresh')] = current_thresh
        
        # 4. THE EXECUTION LOGIC
        if row['Z_Score'] <= -current_thresh and row['VWAP_Distance_Pct'] < 0:
            signals_df.iloc[i, signals_df.columns.get_loc('Signal')] = 'BUY'
            trades_this_hour += 1
            
        elif row['Z_Score'] >= current_thresh and row['VWAP_Distance_Pct'] > 0:
            signals_df.iloc[i, signals_df.columns.get_loc('Signal')] = 'SELL'
            trades_this_hour += 1

    action_log = signals_df[signals_df['Signal'] != 'HOLD']
    return signals_df, action_log

In [93]:
# Accounting Block, aimed at % returns along with drawdowns of the BUY/SELLS signals 


def run_accounting_block(signals_df, initial_capital = 10000.0):
    print(f" Starting Accounting Block (Initial Capital: ${initial_capital:,.2f})")
    
    cash = initial_capital
    position_shares = 0
    entry_price = 0
    
    trade_log = []
    equity_curve = []
    
    # We iterate through the entire timeframe to track our equity at EVERY 15-minute tick.
    # This is required to calculate true Intraday Maximum Drawdown.
    for index, row in signals_df.iterrows():
        current_price = row['Price']
        signal = row['Signal']
        
        # Mark-to-market: Calculate portfolio value at this exact 15m interval
        current_equity = cash + (position_shares * current_price)
        equity_curve.append(current_equity)
        
        # --- EXECUTION LOGIC ---
        if signal == 'BUY' and cash > 0:
            # We go "All In" on the signal (Fractional shares used for exact mathematical tracking)
            position_shares = cash / current_price 
            entry_price = current_price
            cash = 0
            
        elif signal == 'SELL' and position_shares > 0:
            # Liquidate the position
            cash = position_shares * current_price
            dollar_profit = (current_price - entry_price) * position_shares
            trade_profit_pct = ((current_price - entry_price) / entry_price) * 100
            
            # Log the isolated trade (Required for future Monte Carlo stripping)
            trade_log.append({
                'Entry_Time': 'Logged', # Keeping it simple for the array
                'Entry_Price': round(entry_price, 2),
                'Exit_Price': round(current_price, 2),
                'Return_Pct': round(trade_profit_pct, 2),
                'Profit_Dollars': round(dollar_profit, 2)
            })
            
            position_shares = 0
            
    # If the simulation ends while we are still holding shares, we force-liquidate them 
    # to get an accurate final accounting snapshot.
    if position_shares > 0:
        final_price = signals_df.iloc[-1]['Price']
        cash = position_shares * final_price
        dollar_profit = (final_price - entry_price) * position_shares
        trade_log.append({
            'Entry_Time': 'Forced_Close',
            'Entry_Price': round(entry_price, 2),
            'Exit_Price': round(final_price, 2),
            'Return_Pct': round(((final_price - entry_price) / entry_price) * 100, 2),
            'Profit_Dollars': round(dollar_profit, 2)
        })

    # --- ADVANCED METRICS CALCULATION ---
    final_equity = cash
    total_return_pct = ((final_equity - initial_capital) / initial_capital) * 100
    
    # 1. Maximum Drawdown (MDD)
    equity_series = pd.Series(equity_curve)
    rolling_max = equity_series.cummax()
    drawdown = (equity_series - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100
    
    # 2. Win Rate & Profit Factor
    trades_df = pd.DataFrame(trade_log)
    if not trades_df.empty:
        winning_trades = trades_df[trades_df['Return_Pct'] > 0]
        losing_trades = trades_df[trades_df['Return_Pct'] <= 0]
        
        win_rate = (len(winning_trades) / len(trades_df)) * 100
        
        gross_profit = winning_trades['Profit_Dollars'].sum()
        gross_loss = abs(losing_trades['Profit_Dollars'].sum())
        # Profit Factor: How many dollars you make for every dollar you lose
        profit_factor = round(gross_profit / gross_loss, 2) if gross_loss != 0 else float('inf')
    else:
        win_rate = 0
        profit_factor = 0
        
    print(f" Accounting Complete. Final Equity: ${final_equity:,.2f} ({total_return_pct:+.2f}%)")
    
    metrics = {
        'Total_Return_Pct': round(total_return_pct, 2),
        'Max_Drawdown_Pct': round(max_drawdown, 2),
        'Total_Trades': len(trades_df),
        'Win_Rate_Pct': round(win_rate, 2),
        'Profit_Factor': profit_factor
    }
    
    return trades_df, metrics

# === RUN THE ACCOUNTING ON OUR SIGNAL DATA ===
# We pass in the 'evaluated_data' dataframe that came out of our Dynamic Signal Engine
evaluated_data, trade_executions = generate_dynamic_signals(quant_data, target_trades_per_hour=1, base_z_threshold=2.5)

# 2. Now we pass that freshly generated data into the accounting block
trade_history, strategy_metrics = run_accounting_block(evaluated_data)

print("\n--- 📊 STRATEGY METRICS ---")
for key, value in strategy_metrics.items():
    print(f"{key}: {value}")

print("\n--- 📝 INDIVIDUAL TRADE LOG (Top 5) ---")
display(trade_history.head())


 Starting Accounting Block (Initial Capital: $10,000.00)
 Accounting Complete. Final Equity: $9,386.04 (-6.14%)

--- 📊 STRATEGY METRICS ---
Total_Return_Pct: -6.14
Max_Drawdown_Pct: -13.82
Total_Trades: 14
Win_Rate_Pct: 57.14
Profit_Factor: 0.59

--- 📝 INDIVIDUAL TRADE LOG (Top 5) ---


,Entry_Time,Entry_Price,Exit_Price,Return_Pct,Profit_Dollars
0,Logged,625.48,630.56,0.81,81.22
1,Logged,628.29,600.45,-4.43,-446.78
2,Logged,613.63,596.99,-2.71,-261.26
3,Logged,604.17,604.94,0.13,11.95
4,Logged,613.71,610.98,-0.45,-41.79


In [94]:

import itertools

# --- 1. THE UPDATED SIGNAL ENGINE (ENTRIES ONLY) ---
def generate_dynamic_signals(df, target_trades_per_hour=2, base_z_threshold=2.5):
    signals_df = df.copy()
    signals_df['Signal'] = 'HOLD'
    signals_df['Urgency'] = 1.0
    signals_df['Adjusted_Thresh'] = base_z_threshold
    
    trades_this_hour = 0
    current_hour = -1
    
    for i in range(len(signals_df)):
        row = signals_df.iloc[i]
        timestamp = signals_df.index[i]
        
        if timestamp.hour != current_hour:
            trades_this_hour = 0
            current_hour = timestamp.hour
            
        intervals_passed = (timestamp.minute // 15) + 1 
        expected_trades_so_far = (intervals_passed / 4.0) * target_trades_per_hour
        
        urgency = 1.0
        if trades_this_hour < expected_trades_so_far:
            urgency = 1.0 + (expected_trades_so_far - trades_this_hour) * 0.5
        elif trades_this_hour >= target_trades_per_hour:
            urgency = 0.2 
            
        current_thresh = round(base_z_threshold / urgency, 3)
        signals_df.iloc[i, signals_df.columns.get_loc('Urgency')] = round(urgency, 2)
        signals_df.iloc[i, signals_df.columns.get_loc('Adjusted_Thresh')] = current_thresh
        
        # WE ONLY GENERATE BUYS NOW. The Accounting Block handles the Sells.
        # Don't buy after 3:30 PM (no time to make a profit before close)
        if row['Z_Score'] <= -current_thresh and row['VWAP_Distance_Pct'] < 0 and timestamp.hour < 15:
            signals_df.iloc[i, signals_df.columns.get_loc('Signal')] = 'BUY'
            trades_this_hour += 1

    return signals_df, signals_df[signals_df['Signal'] == 'BUY']

# --- 2. THE NEW RISK-MANAGED ACCOUNTING BLOCK ---
def run_accounting_block(signals_df, initial_capital=10000.0, stop_loss_pct=-0.5, take_profit_pct=1.0, max_hold_intervals=6):
    cash = initial_capital
    position_shares = 0
    entry_price = 0
    intervals_held = 0
    
    trade_log = []
    equity_curve = []
    
    for index, row in signals_df.iterrows():
        current_price = row['Price']
        signal = row['Signal']
        timestamp = index
        
        # Mark to market
        current_equity = cash + (position_shares * current_price)
        equity_curve.append(current_equity)
        
        # --- RISK MANAGEMENT EXITS (Checked every 15 mins if holding) ---
        if position_shares > 0:
            intervals_held += 1
            current_return = ((current_price - entry_price) / entry_price) * 100
            
            # The 4 Hard Rules of Path A:
            hit_stop_loss = current_return <= stop_loss_pct
            hit_take_profit = current_return >= take_profit_pct
            hit_time_stop = intervals_held >= max_hold_intervals
            hit_eod = timestamp.hour == 15 and timestamp.minute >= 45 # 3:45 PM
            
            if hit_stop_loss or hit_take_profit or hit_time_stop or hit_eod:
                exit_reason = "Stop Loss" if hit_stop_loss else "Take Profit" if hit_take_profit else "Time Stop" if hit_time_stop else "EOD Close"
                
                cash = position_shares * current_price
                dollar_profit = (current_price - entry_price) * position_shares
                
                trade_log.append({
                    'Exit_Reason': exit_reason,
                    'Entry_Price': round(entry_price, 2),
                    'Exit_Price': round(current_price, 2),
                    'Return_Pct': round(current_return, 2),
                    'Profit_Dollars': round(dollar_profit, 2)
                })
                
                position_shares = 0
                intervals_held = 0
                continue # Skip looking for buys on the exact tick we sell
                
        # --- ENTRY LOGIC ---
        if signal == 'BUY' and position_shares == 0:
            position_shares = cash / current_price 
            entry_price = current_price
            cash = 0
            intervals_held = 0

    # Metrics Compilation
    final_equity = cash if position_shares == 0 else cash + (position_shares * signals_df.iloc[-1]['Price'])
    total_return_pct = ((final_equity - initial_capital) / initial_capital) * 100
    
    equity_series = pd.Series(equity_curve)
    max_drawdown = ((equity_series - equity_series.cummax()) / equity_series.cummax()).min() * 100
    
    trades_df = pd.DataFrame(trade_log)
    if not trades_df.empty:
        winning_trades = trades_df[trades_df['Return_Pct'] > 0]
        losing_trades = trades_df[trades_df['Return_Pct'] <= 0]
        win_rate = (len(winning_trades) / len(trades_df)) * 100
        gross_profit = winning_trades['Profit_Dollars'].sum()
        gross_loss = abs(losing_trades['Profit_Dollars'].sum())
        profit_factor = round(gross_profit / gross_loss, 2) if gross_loss != 0 else float('inf')
    else:
        win_rate = 0
        profit_factor = 0
        
    metrics = {
        'Total_Return_Pct': round(total_return_pct, 2),
        'Max_Drawdown_Pct': round(max_drawdown, 2),
        'Total_Trades': len(trades_df),
        'Win_Rate_Pct': round(win_rate, 2),
        'Profit_Factor': profit_factor
    }
    
    return trades_df, metrics

In [95]:
import itertools

def walk_forward_optimization(df, train_days=20, test_days=5):
    # This function now uses the Path A Risk-Managed Accounting Block
    unique_dates = sorted(list(set(df.index.date)))
    z_thresholds = [1.5, 2.0, 2.5, 3.0]
    target_frequencies = [1, 2, 3, 4]
    
    out_of_sample_results = []
    
    for i in range(0, len(unique_dates) - train_days - test_days, test_days):
        train_start = unique_dates[i]
        train_end = unique_dates[i + train_days]
        test_start = train_end
        test_end = unique_dates[i + train_days + test_days]
        
        train_data = df[(df.index.date >= train_start) & (df.index.date < train_end)]
        test_data = df[(df.index.date >= test_start) & (df.index.date < test_end)]
        
        if train_data.empty or test_data.empty: continue
            
        best_pf = -1
        best_params = (2.5, 2)
        
        # --- TRAINING ---
        for z, freq in itertools.product(z_thresholds, target_frequencies):
            signals, _ = generate_dynamic_signals(train_data, target_trades_per_hour=freq, base_z_threshold=z)
            # Using our new Path A accounting logic
            _, metrics = run_accounting_block(signals)
            
            pf = metrics['Profit_Factor']
            if pf == float('inf'): pf = 99.0
            
            if pf > best_pf and metrics['Total_Trades'] > 0:
                best_pf = pf
                best_params = (z, freq)
                
        # --- OUT-OF-SAMPLE TEST ---
        test_signals, _ = generate_dynamic_signals(test_data, target_trades_per_hour=best_params[1], base_z_threshold=best_params[0])
        _, test_metrics = run_accounting_block(test_signals)
        
        out_of_sample_results.append({
            'Test_Window': f"{test_start}",
            'Trained_Z': best_params[0],
            'Trained_Freq': best_params[1],
            'Blind_Return_%': test_metrics['Total_Return_Pct'],
            'Blind_Drawdown_%': test_metrics['Max_Drawdown_Pct']
        })
        
    oos_df = pd.DataFrame(out_of_sample_results)
    print(f"✅ WFO Complete. Real Avg Return: {oos_df['Blind_Return_%'].mean():.2f}%")
    return oos_df

In [96]:
# Assuming 'quant_data' is still loaded with your BRK-B data
wfo_results = walk_forward_optimization(quant_data)

# Let's also look at the reason the trades are exiting now to verify the math
print("\n--- SAMPLE OF RISK MANAGEMENT EXITS ---")
# Run a quick single pass just to get a sample of the trade log
test_sig, _ = generate_dynamic_signals(quant_data, target_trades_per_hour=2, base_z_threshold=2.5)
test_trades, _ = run_accounting_block(test_sig)
if not test_trades.empty:
    display(test_trades['Exit_Reason'].value_counts())

✅ WFO Complete. Real Avg Return: 0.20%

--- SAMPLE OF RISK MANAGEMENT EXITS ---


Time Stop      24
Stop Loss      14
EOD Close       3
Take Profit     1
Name: Exit_Reason, dtype: int64